<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Image-Dataset-Collector/blob/main/Colab-Image-Dataset-Collector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bulk Image Dataset Collector

Scrape and download image datasets from any webpage to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install requests beautifulsoup4 pillow -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/ImageDatasetCollector/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/ImageDatasetCollector/'  # final destination

SOURCE_URL = 'https://unsplash.com/s/photos/nature'  # webpage URL to scrape images from
MAX_IMAGES = 500  # maximum images to download
MIN_WIDTH = 200  # minimum image width (pixels)
MIN_HEIGHT = 200  # minimum image height (pixels)

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil, glob
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def main():
    import requests
    from bs4 import BeautifulSoup
    from PIL import Image
    import urllib.parse, io
    from IPython.display import display, HTML, Javascript

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print("Keep-alive active")

    if not SOURCE_URL:
        print("ERROR: Set SOURCE_URL in config (e.g. SOURCE_URL = 'https://unsplash.com/s/photos/nature')")
        return

    begin = time.time()
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    r = requests.get(SOURCE_URL, headers=headers, timeout=30)
    soup = BeautifulSoup(r.text, 'html.parser')

    downloaded = 0
    for img in soup.find_all('img'):
        if downloaded >= MAX_IMAGES:
            break
        src = img.get('src') or img.get('data-src') or ''
        if not src or src.startswith('data:'):
            continue
        src = urllib.parse.urljoin(SOURCE_URL, src)
        try:
            ir = requests.get(src, headers=headers, timeout=15)
            pil = Image.open(io.BytesIO(ir.content))
            w, h = pil.size
            if w < MIN_WIDTH or h < MIN_HEIGHT:
                continue
            ext = os.path.splitext(urllib.parse.urlparse(src).path)[1] or '.jpg'
            fname = f'img_{downloaded:05d}{ext}'
            with open(os.path.join(SAVE_PATH, fname), 'wb') as f:
                f.write(ir.content)
            downloaded += 1
            pct = downloaded * 100 // MAX_IMAGES
            bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
            progress_display.update(HTML(
                f'<pre>Downloading: |{bar}| {pct}% | {downloaded}/{MAX_IMAGES} | {fname} ({w}x{h})</pre>'
            ))
        except:
            pass

    print(f'Downloaded {downloaded} images')

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')

    items = glob.glob(os.path.join(SAVE_PATH, '*'))
    items = [f for f in items if os.path.isfile(f)]

    if len(items) > 1:
        import zipfile
        total_sz = sum(os.path.getsize(f) for f in items)
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files ({format_bytes(total_sz)})...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fpath in items:
                zf.write(fpath, os.path.basename(fpath))
                processed += os.path.getsize(fpath)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {os.path.basename(fpath)}</pre>'))
        print(f'Zip: {os.path.basename(zpath)} ({format_bytes(os.path.getsize(zpath))})')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')
    for item in sorted(os.listdir(DRIVE_PATH)):
        print(f'  {item}')
    print('=' * 50)

if __name__ == '__main__':
    main()